In [1]:
from config_info import APIS
from pprint import pprint

from scrapers.arxiv import parser_arxiv
from scrapers.hal import parser_hal
from scrapers.basic_fetching import fetch_raw

# Basic Fetch

# Semantic Scholar

In [7]:
def semantic_fetch(results):
    for result in results["data"]:
        print(result)
        # query = APIS["Semantic Scholar"]["paper_url"].format(paper_id=result["paperId"])
        # res = fetch_raw(query)
        # print(res)
        # print(query)

# CORE

In [4]:
# from dotenv import load_dotenv
# load_dotenv()
# API_KEY = os.getenv("CORE_API_KEY")
# url = "https://api.core.ac.uk/v3/search/works"

# headers = {
#     "Authorization": f"Bearer {API_KEY}"
# }

# params = { 
#     "q": "AI agent",
#     "limit": 5
# }

# r = requests.get(url, headers=headers, params=params)
# r.raise_for_status()

# data = r.json()
# # print(data.keys())
# pprint(data)


# Main 

In [3]:
raw_result = fetch_raw(APIS["arXiv"]["api_url"].format(query="Artificial Intelligence NLP",quantity='10'))
result_arxiv =  parser_arxiv(raw_result)
pprint(result_arxiv) # Arxiv Ok
 
# Test HAL
# url_hal = fetch_raw(APIS["HAL"]["api_url"].format(query="Artificial Intelligence NLP",quantity='10'))
# result_hal = parser_hal(url_hal)
# pprint(result_hal) #Good 

# Test PubMed
# from scrapers.pubmed import fetch_pubmed
# xml_batches_pubmed = fetch_pubmed(
#     query="AI agent",
#     max_results=1,
#     batch_size=1,
#     email="proliquer@scholar-perigueuxu.com"
# )
# pprint(xml_batches_pubmed)

# Test Sementic Scholar
# url_sem = fetch_raw(APIS["Semantic Scholar"]["api_url"].format(query="AI agent",quantity='5'))
# print(type(url_sem))
# semantic_fetch(url_sem)


[{'authors': ['Michael Timothy Bennett', 'Yoshihiro Maruyama'],
  'id': 'http://arxiv.org/abs/2110.01831v1',
  'pdf_url': 'https://arxiv.org/pdf/2110.01831v1',
  'published_at': '2021-10-05T05:58:23Z',
  'source': 'arxiv',
  'summary': 'We attempt to define what is necessary to construct an '
             'Artificial Scientist, explore and evaluate several approaches to '
             'artificial general intelligence (AGI) which may facilitate this, '
             'conclude that a unified or hybrid approach is necessary and '
             'explore two theories that satisfy this requirement to some '
             'degree.',
  'title': 'The Artificial Scientist: Logicist, Emergentist, and Universalist '
           'Approaches to Artificial General Intelligence',
  'updated': '2021-10-05T05:58:23Z'},
 {'authors': ['Michael Timothy Bennett'],
  'id': 'http://arxiv.org/abs/2110.01835v1',
  'pdf_url': 'https://arxiv.org/pdf/2110.01835v1',
  'published_at': '2021-10-05T06:17:02Z',
  'source':

In [11]:
from scrapers.pubmed import format_pubmed_data
pubmed_list = format_pubmed_data(xml_batches_pubmed)


In [4]:
from models.postgres.corpus_schema import metadata, document_table
from config.db_engine import get_db_engine
metadata.create_all(get_db_engine())

ProgrammingError: (psycopg2.errors.InvalidSchemaName) schema "corpus" does not exist
LINE 2: CREATE TABLE corpus.document (
                     ^

[SQL: 
CREATE TABLE corpus.document (
	id VARCHAR NOT NULL, 
	source VARCHAR, 
	source_url VARCHAR, 
	title VARCHAR NOT NULL, 
	summary TEXT, 
	authors JSON, 
	published_at TIMESTAMP WITHOUT TIME ZONE, 
	uri VARCHAR, 
	pdf_url VARCHAR, 
	minio_path VARCHAR, 
	pdf_status VARCHAR, 
	pdf_downloaded_at TIMESTAMP WITHOUT TIME ZONE, 
	pdf_size_bytes BIGINT, 
	pdf_sha256 VARCHAR, 
	created_at TIMESTAMP WITHOUT TIME ZONE DEFAULT now(), 
	updated_at TIMESTAMP WITHOUT TIME ZONE, 
	PRIMARY KEY (id)
)

]
(Background on this error at: https://sqlalche.me/e/20/f405)

In [25]:
from database.postgres.crud import upsert_data
from models.postgres.corpus_schema import document_table 
from config.db_engine import get_db_engine
from processing.cleaning_data import normalize_data

arxiv_mapping = {
    "published_at": "published",
}
raw_result = fetch_raw(APIS["arXiv"]["api_url"].format(query="AI agent",quantity='1'))
result_arxiv =  parser_arxiv(raw_result)
clean_arxiv_data = normalize_data(
    raw_data=result_arxiv, 
    source_name="arXiv",
    date_columns=["published_at"],
    columns_drop=["updated"]
    )


# hal_mapping = { "published" : "published_at"}
# clean_hal_data = normalize_data(
#     raw_data=result_hal,
#     source_name="Hal",
#     column_mapping=hal_mapping,
#     date_columns=["published"]
# )

clean_pubmed_data = normalize_data(
    raw_data= pubmed_list,
    source_name="Pubmed",
    date_columns=["published"]
)
pprint(clean_pubmed_data)
engine = get_db_engine()
# upsert_data(clean_pubmed_data, ['id'], document_table, engine)
# upsert_data(clean_pubmed_data, ['id'], document_table, engine)


[{'authors': ['Theresa Chikopela',
              'Longa Kaluba',
              'Shirley Mwaanga',
              'Fastone M Goma'],
  'id': '1b81b305-9375-5177-a6a8-657a95bf0759',
  'pdf_url': None,
  'published_at': {'Day': '18', 'Month': '04', 'Year': '2026'},
  'source': 'PubMed',
  'summary': {'#text': 'People living with HIV (PLWH) experience a greater '
                       'risk of cardiovascular disease due to HIV-related '
                       'vascular injury. Pulse waveform analysis can '
                       'characterize arterial vascular dysfunction and this '
                       'study compared arterial waveforms in PLWH on '
                       'antiretroviral therapy (ART) with HIV-negative '
                       'controls. In this cross-sectional study, participants '
                       'were recruited from the University Teaching Hospitals, '
                       'Lusaka (September 2018-June 2019). 55 PLWH on ART '
                       '≥2\u2009y